# INFO-H-515 Project: Task 2 - Retrieval and Generation
**Team:** 7

**Objective:** Retrieve relevant course materials using a custom distance metric (Cosine Similarity) and generate grounded exam questions utilizing an external LLM via the Hugging Face Inference API.

In [2]:
# Required installations
# !pip install pyspark sentence-transformers huggingface_hub
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType
from sentence_transformers import SentenceTransformer
import math
from huggingface_hub import InferenceClient
import json
import os
from dotenv import load_dotenv


In [6]:
# Task 2 Parameters
query = "What is the main difference between NoSQL and traditional relational databases?"

embedding_strategy = "SBERT"
similarity_strategy = "Cosine"
number_of_text_chunks_to_retrieve = 5
number_of_questions_to_generate = 3

generator_backend = "HuggingFace_API"
generator_model = "Qwen/Qwen2.5-7B-Instruct"
max_context_tokens = 400

input_dir = "data/data_processed/embedded_chunks.parquet"
output_dir = "data/data_processed/generated_qa.json"

# Credentials
# Load variables from the .env file
load_dotenv()

# Retrieve the token securely from the environment
HF_API_TOKEN = os.getenv("HF_API_TOKEN")

if not HF_API_TOKEN:
    raise ValueError("HF_API_TOKEN not found! Please ensure it is set in your .env file.")



In [ ]:
# Subtask 2.2.1: Information Retrieval
# 1. Initialize Spark Session for Retrieval
spark = SparkSession.builder \
    .appName("BigData_RAG_Task2_Retrieval") \
    .getOrCreate()

# 2. Load the vector database
df_chunks = spark.read.parquet(input_dir)

# 3. Load the embedding model
print(f"Loading {embedding_strategy} model to embed the query...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
query_embedding = embedder.encode(query).tolist()

# 4. Manual Implementation of Cosine Similarity
def get_cosine_similarity_udf(query_vec):
    """
    Computes the cosine similarity between the user query vector 
    and the chunk embeddings stored in the dataframe.
    """
    def compute_similarity(chunk_vec):
        if not chunk_vec or not query_vec:
            return 0.0
            
        dot_product = sum(a * b for a, b in zip(query_vec, chunk_vec))
        norm_a = math.sqrt(sum(a * a for a in query_vec))
        norm_b = math.sqrt(sum(b * b for b in chunk_vec))
        
        if norm_a == 0 or norm_b == 0:
            return 0.0
            
        return dot_product / (norm_a * norm_b)
        
    return udf(compute_similarity, DoubleType())

# 5. Apply the similarity metric and retrieve the Top-N chunks
df_with_scores = df_chunks.withColumn(
    "similarity_score", 
    get_cosine_similarity_udf(query_embedding)(col("embedding"))
)

top_chunks_df = df_with_scores.orderBy(col("similarity_score").desc()).limit(number_of_text_chunks_to_retrieve)

# 6. Collect results to the local driver to avoid Spark executor issues during LLM generation
retrieved_results = top_chunks_df.select(
    "chunk_id", "similarity_score", "chunk_text", "page_num", "source_pdf"
).collect()

print(f"Successfully retrieved Top {number_of_text_chunks_to_retrieve} chunks.")

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 16:09:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loading SBERT model to embed the query...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully retrieved Top 5 chunks.


In [ ]:
# Subtask 2.2.2: Generation and LLM Integration
# 1. Initialize the Hugging Face Inference Client
# Using Qwen 2.5 (7B) as it exceeds the baseline requirements (llama3.2:3b) 
client = InferenceClient(model=generator_model, token=HF_API_TOKEN)

# 2. Build the Grounded Prompt dynamically using the retrieved chunks
sources_text = ""
for rank, row in enumerate(retrieved_results, start=1):
    # Enforcing the exact citation format required by the project guidelines
    sources_text += f"[{rank}] (source: {row.source_pdf}, page {row.page_num}, chunk {row.chunk_id}) {row.chunk_text}\n\n"

# --- DYNAMIC ADJUSTMENTS FOR MULTIPLE QUESTIONS ---
# Adjust the maximum tokens dynamically based on the requested number of questions
# Allocating approximately 350 tokens per question
dynamic_max_tokens = 350 * number_of_questions_to_generate

# Adjust the prompt wording for singular/plural
plural_s = "s" if number_of_questions_to_generate > 1 else ""
each_word = "EACH" if number_of_questions_to_generate > 1 else "the"

# System and User instructions for the LLM
messages = [
    {
        "role": "system", 
        "content": f'You are an expert university professor. Your task is to generate exactly {number_of_questions_to_generate} high-quality exam question{plural_s} based ONLY on the provided sources. If the sources do not contain enough information, say: "I cannot generate a question based on these sources."'
    },
    {
        "role": "user", 
        "content": f"""SOURCES:
{sources_text}

For {each_word} question, you MUST provide:
1. The Question (Q)
2. The Expected Answer (A)
3. The Supporting Citation(s) using the bracketed numbers like [1] or [2].

Output format:
Question 1:
Q: [Question text]
A: [Answer Key]
Citation: [Source Number(s)]

Question 2: (if applicable)
...and so on for all {number_of_questions_to_generate} question{plural_s}."""
    }
]

print(f"--- Sending Prompt to {generator_model} via {generator_backend} ---")

# 3. Call the API and generate the question
try:
    # Setting temperature to 0.0 for strict reproducibility as required
    # Using the dynamically calculated max_tokens
    response = client.chat_completion(
        messages=messages, 
        max_tokens=dynamic_max_tokens, 
        temperature=0.0  
    )
    
    generated_qa = response.choices[0].message.content
    print("\n--- LLM Generated Output ---\n")
    print(generated_qa)
    
    # 4. Save results to disk for traceability and evaluation
    # Structure the data to be saved
    output_data = {
        "user_query": query,
        "exact_prompt_sent": messages,
        "generation_parameters": {
            "model": generator_model,
            "backend": generator_backend,
            "temperature": 0.0,
            "max_tokens": dynamic_max_tokens # Reflecting the dynamic limit
        },
        "retrieved_chunks_used": [row.chunk_id for row in retrieved_results],
        "llm_output": generated_qa
    }
    
    # Ensure the output directory exists
    os.makedirs(os.path.dirname(output_dir), exist_ok=True)
    
    # Write the JSON file
    with open(output_dir, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)
        
    print(f"\n✅ SUCCESS: Generation saved and documented in {output_dir}")

except Exception as e:
    print(f"\nError during LLM generation: {e}")

--- Sending Prompt to Qwen/Qwen2.5-7B-Instruct via HuggingFace_API ---

--- LLM Generated Output ---

Question 1:
Q: What does the term "NoSQL" practically mean, and how was it introduced?
A: The term "NoSQL" practically means "not only SQL." It was introduced accidentally as a hashtag for a gathering.
Citation: [1]

Question 2:
Q: What are the common characteristics of NoSQL databases, and how do they manage large-scale data?
A: Common characteristics of NoSQL databases include being non-relational, cluster-friendly (not all), open source (most), and designed to manage large-scale data, inspired by modern Web challenges. They are schema-less and can handle large volumes of data.
Citation: [1]

Question 3:
Q: What are the four main types of NoSQL databases mentioned in the document, and provide an example of each type?
A: The four main types of NoSQL databases are key-value stores, document stores, column families, and graph databases. Examples include:
- Key-value stores: Redis
- Docu